# mCREAM Graph Module Ensemble Analysis

Architecture: M Concept-Concept blocks (one per expert graph), shared backbone/side-channel/task-head.
Aggregation at **concept level** `c = mean(c_0..c_M)`, not prediction level.

## Three experiment types:
1. **Noisy DAGs** — same action/level, different random seeds per expert
2. **Single-edge perturbation** — each expert missing/adding exactly 1 edge
3. **Mixed noise levels** — experts from low/medium/high, λ can learn


In [ ]:
import pandas as pd, numpy as np, ast, torch
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings; warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

ACTION_COLOR  = {'deletion':'#e74c3c','addition':'#2ecc71','reversal':'#3498db'}
LEVEL_ALPHA   = {'low':0.45,'medium':0.70,'high':1.0}
LEVEL_COLOR   = {'low':'#3498db','medium':'#e67e22','high':'#e74c3c'}

EXPERIMENTS_ROOT = Path('/home/dani00003/mCREAM/experiments')
GRAPHS_ROOT      = Path('/home/dani00003/mCREAM/data/FashionMNIST')
DAG_CFMNIST      = GRAPHS_ROOT / 'Complete_Concept_FMNIST_DAG.csv'

DATASETS = ['Complete_Concept_FMNIST']
ACTIONS  = ['deletion','addition','reversal']
LEVELS   = ['low','medium','high']
N_EXPERTS = 5

# ── Shared helpers ────────────────────────────────────────────────────────────
def load_gt_cream(root):
    rows=[]
    for ds,model,exp in [('Complete_Concept_FMNIST','Standard_FashionMNIST','CREAM_best_cfmnist')]:
        md=root/ds/'train_cbm'/model/exp/'last_metrics'
        if not md.exists(): continue
        for f in sorted(md.glob('*.csv')):
            df=pd.read_csv(f); df['dataset']=ds; rows.append(df)
    gt=pd.concat(rows,ignore_index=True) if rows else pd.DataFrame()
    if len(gt)>0:
        r=gt.iloc[0]
        print(f'GT CREAM: acc={r.get("test_task_accuracy",float("nan")):.4f}  '
              f'concept={r.get("test_concept_accuracy",float("nan")):.4f}  CCI={r.get("CCI","N/A")}')
    return gt

def load_interventions(root, exp_name_filter=None):
    rows=[]
    for ds in DATASETS:
        ens_dir=root/ds/'train_cbm'/'mCREAM_GraphEnsemble'
        if not ens_dir.exists(): continue
        for csv_f in ens_dir.rglob('intervention_results.csv'):
            try:
                df=pd.read_csv(csv_f)
                parts=csv_f.parts
                exp_name=seed=None
                for i,p in enumerate(parts):
                    if p=='mCREAM_GraphEnsemble' and i+1<len(parts): exp_name=parts[i+1]
                    if p.startswith('seed_'): seed=int(p.split('_')[1])
                if exp_name is None: continue
                if exp_name_filter and not any(f in exp_name for f in exp_name_filter): continue
                df['dataset']=ds; df['exp_name']=exp_name; df['seed']=seed
                rows.append(df)
            except: pass
    return pd.concat(rows,ignore_index=True) if rows else pd.DataFrame()

def show_expert_dags(dirs_and_indices, title, K=11):
    """Show expert graphs used in one experiment."""
    gt_df=pd.read_csv(DAG_CFMNIST,index_col=0)
    gt_vals=(gt_df.values!=0).astype(int)
    gt_u2c=gt_vals[:K,:K]

    M=len(dirs_and_indices)
    fig,axes=plt.subplots(1,M+1,figsize=(2.5*(M+1),3))
    fig.suptitle(title,fontsize=10,fontweight='bold')

    def make_rgb(p):
        rgb=np.zeros((*p.shape,3))
        for r in range(p.shape[0]):
            for c in range(p.shape[1]):
                if   gt_u2c[r,c]==1 and p[r,c]==1: rgb[r,c]=[0.2,0.4,0.8]
                elif gt_u2c[r,c]==0 and p[r,c]==1: rgb[r,c]=[0.2,0.8,0.2]
                elif gt_u2c[r,c]==1 and p[r,c]==0: rgb[r,c]=[0.9,0.2,0.2]
                else: rgb[r,c]=[0.93,0.93,0.93]
        return rgb

    # GT
    gt_rgb=np.zeros((*gt_u2c.shape,3))
    for r in range(K):
        for c in range(K):
            gt_rgb[r,c]=[0.2,0.4,0.8] if gt_u2c[r,c]==1 else [0.93,0.93,0.93]
    axes[0].imshow(gt_rgb,aspect='auto',interpolation='nearest')
    axes[0].set_title('GT',fontweight='bold',fontsize=8)
    for sp in axes[0].spines.values(): sp.set_edgecolor('gold'); sp.set_linewidth(3)
    axes[0].set_xticks([]); axes[0].set_yticks([])

    for m,(graph_dir,idx,label) in enumerate(dirs_and_indices):
        f=Path(graph_dir)/'u2c'/f'expert_{idx}.pt'
        ax=axes[m+1]
        if not f.exists(): ax.text(0.5,0.5,'N/A',ha='center',va='center',transform=ax.transAxes,fontsize=8); ax.axis('off'); continue
        p=torch.load(f,weights_only=True).float().numpy()
        ax.imshow(make_rgb(p),aspect='auto',interpolation='nearest')
        ax.set_title(label,fontsize=7)
        ax.set_xticks([]); ax.set_yticks([])

    from matplotlib.patches import Patch
    fig.legend(handles=[Patch(facecolor=[0.2,0.4,0.8],label='kept'),
                         Patch(facecolor=[0.9,0.2,0.2],label='deleted'),
                         Patch(facecolor=[0.2,0.8,0.2],label='added')],
               loc='lower center',ncol=3,fontsize=7,bbox_to_anchor=(0.5,-0.05))
    plt.tight_layout(rect=[0,0.05,1,1])
    plt.show()

gt_df = load_gt_cream(EXPERIMENTS_ROOT)
print(f'Root exists: {EXPERIMENTS_ROOT.exists()}')


---
# Experiment 1 — Noisy DAGs (same level, different seeds)

M=5 experts, each with same noise type+level but different random seed → different specific edges.

## Two intervention modes (shown separately below)

**Individual concept interventions** (`group_interventions=False`):
- Replace one concept at a time, randomly chosen
- x-axis goes 0 → 11 (one per concept)
- Answers: *how much does each extra concept fix help?*

**Group interventions** (`group_interventions=True`):
- Replace all concepts in one mutex group at once (cfmnist has 3 groups)
- x-axis goes 0 → 3 (one per mutex group)
- One step = revealing a full mutually-exclusive set (e.g. all Season concepts together)
- Answers: *how much does fixing a whole semantic group help?*
- Steeper curve = model relies more on that semantic group for prediction

In [ ]:
def load_noisy_dag_results(root):
    rows=[]
    for ds in DATASETS:
        ens_dir=root/ds/'train_cbm'/'mCREAM_GraphEnsemble'
        if not ens_dir.exists(): continue
        for exp_dir in sorted(ens_dir.iterdir()):
            if not exp_dir.is_dir(): continue
            name=exp_dir.name
            if not name.startswith('graph_ensemble_') or 'mixed' in name or 'gensemble' in name: continue
            p=name.split('_')
            try: action=p[2]; level=p[3]
            except: continue
            for seed_dir in sorted(exp_dir.glob('seed_*/lightning_logs/version_*')):
                seed=int(seed_dir.parent.parent.name.split('_')[1])
                for csv_f in sorted(seed_dir.glob('*.csv')):
                    if any(x in csv_f.name for x in ['perc_','_set_','intervention','exogenous']): continue
                    try:
                        df=pd.read_csv(csv_f)
                        df['dataset']=ds;df['action']=action;df['noise_level']=level;df['seed']=seed
                        rows.append(df)
                    except: pass
    if not rows: print('No noisy DAG results'); return pd.DataFrame()
    df=pd.concat(rows,ignore_index=True)
    print(f'Exp1 loaded: {len(df)} rows | actions={sorted(df.action.unique())} | levels={sorted(df.noise_level.unique())}')
    return df

exp1_df    = load_noisy_dag_results(EXPERIMENTS_ROOT)
exp1_interv= load_interventions(EXPERIMENTS_ROOT, exp_name_filter=['graph_ensemble_deletion','graph_ensemble_addition','graph_ensemble_reversal'])


In [ ]:
# Show expert graphs for each action/level
graphs_base = GRAPHS_ROOT / 'expert_graphs' / 'ensemble'
for action in ACTIONS:
    for level in LEVELS:
        d = graphs_base / f'{action}_{level}'
        if not d.exists(): continue
        dirs_and_indices = [(d, m, f'E{m}\n{level}') for m in range(N_EXPERTS)]
        show_expert_dags(dirs_and_indices, f'Exp1: {action}/{level} — expert graphs used')


In [ ]:
if len(exp1_df)==0: print('No Exp1 data')
else:
    lnum={'low':0.25,'medium':0.50,'high':0.75}
    KEY=['test_task_accuracy','test_concept_accuracy','CCI','PFI_concept_importance']
    av=[c for c in KEY if c in exp1_df.columns]
    means=exp1_df.groupby(['action','noise_level'])[av].mean()
    stds =exp1_df.groupby(['action','noise_level'])[av].std()
    sm=pd.DataFrame(index=means.index)
    for c in av: sm[c]=means[c].map('{:.4f}'.format)+' +/- '+stds[c].map('{:.4f}'.format)
    if len(gt_df)>0:
        print('GT CREAM baseline:')
        for c in av:
            if c in gt_df.columns: print(f'  {c:30s} {gt_df[c].mean():.4f}')
    print('\n--- Experiment 1: Noisy DAGs ---')
    display(sm)

    # Accuracy vs noise level
    fig,axes=plt.subplots(1,2,figsize=(12,4.5))
    fig.suptitle('Exp1: Noisy DAGs — accuracy vs noise level',fontsize=12,fontweight='bold')
    for ax,metric,ylabel in [(axes[0],'test_task_accuracy','Task Accuracy'),(axes[1],'test_concept_accuracy','Concept Accuracy')]:
        if metric not in exp1_df.columns: continue
        d=exp1_df.copy(); d['noise_prob']=d['noise_level'].map(lnum)
        agg=d.groupby(['action','noise_prob'])[metric].agg(['mean','std']).reset_index()
        for action in ACTIONS:
            sub=agg[agg['action']==action]
            if sub.empty: continue
            col=ACTION_COLOR[action]
            ax.plot(sub['noise_prob'],sub['mean'],color=col,marker='o',markersize=8,lw=2,label=action)
            ax.fill_between(sub['noise_prob'],sub['mean']-sub['std'],sub['mean']+sub['std'],alpha=0.12,color=col)
        if len(gt_df)>0 and metric in gt_df.columns:
            v=gt_df[metric].mean()
            ax.axhline(v,color='black',lw=2,ls='--',label=f'GT CREAM: {v:.4f}')
        ax.set_xlabel('Noise probability'); ax.set_ylabel(ylabel)
        ax.set_xticks([0.25,0.50,0.75]); ax.set_xticklabels(['low','medium','high'])
        ax.legend(title='Noise type',fontsize=8)
    plt.tight_layout(); plt.savefig('exp1_accuracy.png',dpi=150,bbox_inches='tight'); plt.show()

    # ── Intervention curves: TWO plots side by side per action ────────────────
    if len(exp1_interv)>0:
        for action in ACTIONS:
            sub_all = exp1_interv[exp1_interv['exp_name'].str.contains(action)]
            if sub_all.empty: continue

            # Split into individual and group
            has_group_col = 'group_interventions' in sub_all.columns
            sub_indiv = sub_all[sub_all['group_interventions']==False] if has_group_col else sub_all
            sub_group = sub_all[sub_all['group_interventions']==True]  if has_group_col else pd.DataFrame()

            fig, axes = plt.subplots(2, len(LEVELS), figsize=(5*len(LEVELS), 9), sharey='row')
            fig.suptitle(
                f'Exp1: {action} — Intervention curves\n'
                f'TOP: individual concept (0→11)   BOTTOM: group/mutex (0→3)',
                fontsize=11, fontweight='bold'
            )

            for col_idx, level in enumerate(LEVELS):
                # ── TOP ROW: individual interventions ─────────────────────────
                ax_top = axes[0][col_idx]
                lv = sub_indiv[sub_indiv['exp_name'].str.contains(level)]
                if not lv.empty:
                    agg = lv.groupby('num_interventions')['test_task_accuracy'].agg(['mean','std']).reset_index()
                    ax_top.plot(agg['num_interventions'], agg['mean'],
                                color=ACTION_COLOR[action], lw=2.5, marker='o', markersize=5)
                    ax_top.fill_between(agg['num_interventions'],
                                        agg['mean']-agg['std'], agg['mean']+agg['std'],
                                        alpha=0.15, color=ACTION_COLOR[action])
                ax_top.set_title(f'{level} — individual', fontsize=10)
                ax_top.set_xlabel('Concepts replaced (0→11)')
                ax_top.set_ylabel('Task Accuracy' if col_idx==0 else '')
                ax_top.tick_params(labelsize=8)

                # ── BOTTOM ROW: group interventions ───────────────────────────
                ax_bot = axes[1][col_idx]
                lv_g = sub_group[sub_group['exp_name'].str.contains(level)]
                if not lv_g.empty:
                    agg_g = lv_g.groupby('num_interventions')['test_task_accuracy'].agg(['mean','std']).reset_index()
                    ax_bot.plot(agg_g['num_interventions'], agg_g['mean'],
                                color=ACTION_COLOR[action], lw=2.5, marker='s', markersize=6,
                                ls='--')
                    ax_bot.fill_between(agg_g['num_interventions'],
                                        agg_g['mean']-agg_g['std'], agg_g['mean']+agg_g['std'],
                                        alpha=0.15, color=ACTION_COLOR[action])
                ax_bot.set_title(f'{level} — group (mutex)', fontsize=10)
                ax_bot.set_xlabel('Mutex groups replaced (0→3)')
                ax_bot.set_ylabel('Task Accuracy' if col_idx==0 else '')
                ax_bot.tick_params(labelsize=8)

            plt.tight_layout()
            plt.savefig(f'exp1_interv_{action}.png', dpi=150, bbox_inches='tight')
            plt.show()

---
# Experiment 2 — Edge Count ±5 from GT

M=5 experts, each with different seed's graph of same u2c edge count.
x-axis: edge count (12..17GT..22), y-axis: accuracy/CCI.
Compare: how does the graph ensemble perform as u2c edge count varies?


In [ ]:
def load_edge_count_results(root):
    """Load Exp2: edge count experiments.
    Folder names: gensemble_edge_count_u2c_12edges, ..._17edges (GT), ..._22edges
    """
    rows=[]
    for ds in DATASETS:
        ens_dir=root/ds/'train_cbm'/'mCREAM_GraphEnsemble'
        if not ens_dir.exists(): print(f'  [SKIP] {ens_dir}'); continue
        for exp_dir in sorted(ens_dir.iterdir()):
            if not exp_dir.is_dir(): continue
            name=exp_dir.name
            if not name.startswith('gensemble_edge_count_u2c_'): continue
            # Parse edge count: gensemble_edge_count_u2c_12edges
            import re
            m=re.match(r'gensemble_edge_count_(u2c|c2y)_(\d+)edges', name)
            if not m: continue
            graph_type=m.group(1); n_edges=int(m.group(2))
            for seed_dir in sorted(exp_dir.glob('seed_*/lightning_logs/version_*')):
                seed=int(seed_dir.parent.parent.name.split('_')[1])
                for csv_f in sorted(seed_dir.glob('*.csv')):
                    if any(x in csv_f.name for x in ['perc_','_set_','intervention','exogenous']): continue
                    try:
                        df=pd.read_csv(csv_f)
                        df['dataset']=ds;df['graph_type']=graph_type
                        df['edge_count']=n_edges;df['seed']=seed
                        rows.append(df)
                    except: pass
    if not rows: print('No Exp2 edge count data'); return pd.DataFrame()
    df=pd.concat(rows,ignore_index=True)
    print(f'Exp2 loaded: {len(df)} rows')
    print(f'  edge counts: {sorted(df.edge_count.unique())}')
    print(f'  GT=17 is at x=17 (red dotted line)')
    return df

exp2_df = load_edge_count_results(EXPERIMENTS_ROOT)


In [ ]:
# Show expert graphs for a specific edge count
# Each expert has different random edges removed/added at the same total count
edge_counts_to_show = [12, 17, 22]  # GT-5, GT, GT+5
graphs_base = GRAPHS_ROOT / 'expert_graphs' / 'graph_ensemble_edge_count'

for count in edge_counts_to_show:
    d = graphs_base / f'u2c_{count}edges'
    if not d.exists(): print(f'Graphs not found: {d}'); continue
    dirs_and_indices = [(d, m, f'E{m}
seed{m}') for m in range(N_EXPERTS)]
    show_expert_dags(dirs_and_indices,
                     f'Edge count={count} u2c edges (GT=17)
'
                     f'Each expert has different random edges removed/added')


In [ ]:
if len(exp2_df)==0:
    print('No Exp2 data yet.')
    print('Submit: bash server_scripts/mcream_experiment/submit_graph_ensemble_edge_count.sh --u2c-only')
else:
    GT_COUNT = 17
    TYPE_COLOR = {'u2c':'#3498db','c2y':'#e74c3c'}

    for metric, ylabel in [('test_task_accuracy','Task Accuracy'),
                            ('test_concept_accuracy','Concept Accuracy'),
                            ('CCI','CCI')]:
        if metric not in exp2_df.columns: continue
        for ds in DATASETS:
            ds_df=exp2_df[exp2_df['dataset']==ds].dropna(subset=[metric])
            if ds_df.empty: continue
            graph_types=sorted(ds_df['graph_type'].unique())
            fig,axes=plt.subplots(1,len(graph_types),figsize=(7*len(graph_types),5),sharey=False)
            if len(graph_types)==1: axes=[axes]
            fig.suptitle(f'{ds}\nExp2: Edge Count vs {ylabel}\n'
                         f'Each point = mean ± std across 5 training seeds',
                         fontsize=11,fontweight='bold')
            for ax,gtype in zip(axes,graph_types):
                sub=ds_df[ds_df['graph_type']==gtype]
                edge_counts=sorted(sub['edge_count'].unique())
                color=TYPE_COLOR.get(gtype,'#3498db')
                agg=sub.groupby('edge_count')[metric].agg(['mean','std']).reset_index()
                ax.plot(agg['edge_count'],agg['mean'],color=color,marker='o',markersize=8,lw=2)
                ax.fill_between(agg['edge_count'],agg['mean']-agg['std'],agg['mean']+agg['std'],alpha=0.15,color=color)
                ax.axvline(x=GT_COUNT,color='red',ls=':',lw=2,alpha=0.7,label=f'GT count ({GT_COUNT})')
                if len(gt_df)>0 and metric in gt_df.columns:
                    sub2=gt_df[gt_df['dataset']==ds]
                    if not sub2.empty and pd.notna(sub2[metric].mean()):
                        v=sub2[metric].mean()
                        ax.axhline(v,color='black',lw=2,ls='--',label=f'GT CREAM: {v:.4f}')
                ax.set_title(f'{gtype.upper()} ({"concept→concept" if gtype=="u2c" else "concept→task"})',fontsize=10)
                ax.set_xlabel('Number of u2c edges',fontsize=9)
                ax.set_ylabel(ylabel,fontsize=9)
                ax.set_xticks(edge_counts); ax.tick_params(labelsize=8)
                ax.legend(fontsize=8,framealpha=0.9)
            plt.tight_layout()
            plt.savefig(f'exp2_edge_count_{metric}_{ds}.png',dpi=150,bbox_inches='tight')
            plt.show()

    # ── Intervention curves: individual and group shown separately ─────────────
    exp2_interv = load_interventions(EXPERIMENTS_ROOT, exp_name_filter=['gensemble_edge_count_u2c'])

    if len(exp2_interv) > 0:
        import re
        exp2_interv['edge_count'] = exp2_interv['exp_name'].apply(
            lambda x: int(re.search(r'(\d+)edges', x).group(1))
            if re.search(r'(\d+)edges', x) else None)
        exp2_interv = exp2_interv.dropna(subset=['edge_count'])
        exp2_interv['edge_count'] = exp2_interv['edge_count'].astype(int)

        has_group_col = 'group_interventions' in exp2_interv.columns
        sub_indiv = exp2_interv[exp2_interv['group_interventions']==False] if has_group_col else exp2_interv
        sub_group = exp2_interv[exp2_interv['group_interventions']==True]  if has_group_col else pd.DataFrame()

        cmap = plt.cm.RdYlGn
        for ds in DATASETS:
            show_counts = [c for c in [12, 15, 17, 19, 22] if c in sub_indiv['edge_count'].unique()]
            if not show_counts: show_counts = sorted(sub_indiv['edge_count'].unique())
            all_counts = sorted(sub_indiv['edge_count'].unique())

            fig, axes = plt.subplots(1, 2, figsize=(14, 5))
            fig.suptitle(
                f'{ds}\nExp2: Intervention Curves per Edge Count (GT=17, band = ±std across 5 seeds)\n'
                f'LEFT: individual concept (0→11)   RIGHT: group/mutex (0→3)',
                fontsize=11, fontweight='bold'
            )

            for ax, src, xlabel, title_suffix in [
                (axes[0], sub_indiv, 'Concepts replaced (0→11)',  'Individual interventions'),
                (axes[1], sub_group, 'Mutex groups replaced (0→3)', 'Group interventions'),
            ]:
                ds_src = src[src['dataset']==ds] if 'dataset' in src.columns else src
                if ds_src.empty:
                    ax.set_title(f'{title_suffix} — no data'); ax.axis('off'); continue
                sc = [c for c in show_counts if c in ds_src['edge_count'].unique()]
                for count in sc:
                    lv  = ds_src[ds_src['edge_count']==count]
                    agg = lv.groupby('num_interventions')['test_task_accuracy'].agg(['mean','std']).reset_index()
                    norm  = all_counts.index(count) / max(len(all_counts)-1, 1)
                    color = cmap(norm)
                    style = '--' if count < GT_COUNT else ('-' if count==GT_COUNT else '-.')
                    lw    = 3 if count==GT_COUNT else 1.8
                    ax.plot(agg['num_interventions'], agg['mean'],
                            color=color, lw=lw, ls=style, marker='o', markersize=4,
                            label=f'{count}{"(GT)" if count==GT_COUNT else ""}')
                    ax.fill_between(agg['num_interventions'],
                                    agg['mean']-agg['std'], agg['mean']+agg['std'],
                                    alpha=0.08, color=color)
                ax.axhline(1.0, color='gray', ls=':', alpha=0.4, lw=1)
                ax.set_title(title_suffix, fontsize=10)
                ax.set_xlabel(xlabel, fontsize=9)
                ax.set_ylabel('Task Accuracy', fontsize=9)
                ax.legend(fontsize=7, loc='lower right', framealpha=0.95)
                ax.tick_params(labelsize=8)

            plt.tight_layout()
            plt.savefig(f'exp2_interventions_{ds}.png', dpi=150, bbox_inches='tight')
            plt.show()
    else:
        print('No intervention results for Exp2 yet.')

---
# Experiment 3 — Mixed Noise Levels

Experts from different noise levels (low/medium/high) in ONE model.
λ_m should learn: high-noise experts get higher weight (harder to train).


In [ ]:
def load_mixed_level_results(root):
    """Load Exp3: mixed noise levels.
    Folder names: graph_ensemble_mixed_deletion, graph_ensemble_mixed_addition, etc.
    """
    rows=[]
    for ds in DATASETS:
        ens_dir=root/ds/'train_cbm'/'mCREAM_GraphEnsemble'
        if not ens_dir.exists(): continue
        for exp_dir in sorted(ens_dir.iterdir()):
            if not exp_dir.is_dir(): continue
            name=exp_dir.name
            if 'mixed' not in name or not name.startswith('graph_ensemble_mixed'): continue
            # Extract action: graph_ensemble_mixed_{action}
            action=name.replace('graph_ensemble_mixed_','')
            for seed_dir in sorted(exp_dir.glob('seed_*/lightning_logs/version_*')):
                seed=int(seed_dir.parent.parent.name.split('_')[1])
                for csv_f in sorted(seed_dir.glob('*.csv')):
                    if any(x in csv_f.name for x in ['perc_','_set_','intervention','exogenous']): continue
                    try:
                        df=pd.read_csv(csv_f)
                        df['dataset']=ds;df['action']=action;df['seed']=seed
                        rows.append(df)
                    except: pass
    if not rows: print('No Exp3 data yet'); return pd.DataFrame()
    df=pd.concat(rows,ignore_index=True)
    print(f'Exp3 loaded: {len(df)} rows | actions={sorted(df.action.unique())}')
    return df

exp3_df     = load_mixed_level_results(EXPERIMENTS_ROOT)
exp3_interv = load_interventions(EXPERIMENTS_ROOT, exp_name_filter=['graph_ensemble_mixed'])


In [ ]:
# Show which expert graphs are used in mixed experiment
graphs_base = GRAPHS_ROOT / 'expert_graphs' / 'ensemble'
for action in ACTIONS:
    # Mixed assignment: low/0, medium/0, high/0, low/1, high/1
    dirs_and_indices = [
        (graphs_base/f'{action}_low',    0, f'E0\nlow'),
        (graphs_base/f'{action}_medium', 0, f'E1\nmedium'),
        (graphs_base/f'{action}_high',   0, f'E2\nhigh'),
        (graphs_base/f'{action}_low',    1, f'E3\nlow'),
        (graphs_base/f'{action}_high',   1, f'E4\nhigh'),
    ]
    show_expert_dags(dirs_and_indices, f'Exp3 mixed {action}: expert graphs (different noise levels)')


In [ ]:
if len(exp3_df)==0: print('No Exp3 data yet. Submit: bash submit_graph_ensemble.sh then run mixed configs')
else:
    KEY=['test_task_accuracy','test_concept_accuracy','CCI']
    av=[c for c in KEY if c in exp3_df.columns]
    print('--- Experiment 3: Mixed Noise Levels ---')
    display(exp3_df.groupby('action')[av].mean().round(4))
    # Lambda weights — the key result for Exp3
    lambda_cols=[c for c in exp3_df.columns if c.startswith('lambda_')]
    if lambda_cols:
        print('\nLearned lambda weights per expert (0=low noise, 2=high noise):')
        lam=exp3_df.groupby('action')[lambda_cols].mean().round(4)
        lam.columns=[f'E{c.replace("lambda_","")}' for c in lam.columns]
        display(lam)
        # Bar chart of lambdas
        fig,axes=plt.subplots(1,len(exp3_df['action'].unique()),figsize=(5*len(exp3_df['action'].unique()),4))
        if not hasattr(axes,'__len__'): axes=[axes]
        expert_labels=['E0\n(low)','E1\n(med)','E2\n(high)','E3\n(low)','E4\n(high)']
        for ax,action in zip(axes,sorted(exp3_df['action'].unique())):
            sub=exp3_df[exp3_df['action']==action][lambda_cols].mean()
            colors=['#3498db','#e67e22','#e74c3c','#3498db','#e74c3c']
            ax.bar(range(len(sub)),sub.values,color=colors,alpha=0.8)
            ax.axhline(0.2,color='black',ls='--',lw=1.5,alpha=0.5,label='uniform (0.2)')
            ax.set_xticks(range(len(sub))); ax.set_xticklabels(expert_labels,fontsize=8)
            ax.set_title(action,fontsize=10); ax.set_ylabel('Lambda weight')
            ax.legend(fontsize=7)
        fig.suptitle('Exp3: Learned lambda per expert\n(blue=low noise, red=high noise)',fontsize=11,fontweight='bold')
        plt.tight_layout(); plt.savefig('exp3_lambdas.png',dpi=150,bbox_inches='tight'); plt.show()
    # Intervention curves
    if len(exp3_interv)>0:
        fig,axes=plt.subplots(1,len(ACTIONS),figsize=(5*len(ACTIONS),4.5),sharey=True)
        fig.suptitle('Exp3: Mixed levels — intervention curves',fontsize=11,fontweight='bold')
        for ax,action in zip(axes,ACTIONS):
            sub=exp3_interv[exp3_interv['exp_name'].str.contains(f'mixed_{action}')]
            if sub.empty: ax.set_title(action); continue
            agg=sub.groupby('num_interventions')['test_task_accuracy'].agg(['mean','std']).reset_index()
            ax.plot(agg['num_interventions'],agg['mean'],color=ACTION_COLOR[action],lw=2.5,marker='o',markersize=5)
            ax.fill_between(agg['num_interventions'],agg['mean']-agg['std'],agg['mean']+agg['std'],alpha=0.15,color=ACTION_COLOR[action])
            ax.set_title(action,fontsize=10); ax.set_xlabel('Interventions'); ax.tick_params(labelsize=8)
            ax.set_ylabel('Task Accuracy' if action==ACTIONS[0] else '')
        plt.tight_layout(); plt.savefig('exp3_interventions.png',dpi=150,bbox_inches='tight'); plt.show()


---
# Learning Curves — Training Loss per Epoch

Loads `metrics.csv` from each `seed_*/lightning_logs/version_*/` directory.
Shows `train_task_loss`, `train_concept_loss`, `train_total_loss` across epochs.

Useful for:
- Diagnosing convergence
- Explaining accuracy variance across seeds (do some seeds diverge early?)
- Showing supervisor that training is stable

In [ ]:
import re as _re

def load_learning_curves(root, exp_name_filter=None):
    """Load per-epoch metrics.csv for all seeds of matching experiments."""
    rows = []
    for ds in DATASETS:
        ens_dir = root / ds / 'train_cbm' / 'mCREAM_GraphEnsemble'
        if not ens_dir.exists(): continue
        for exp_dir in sorted(ens_dir.iterdir()):
            if not exp_dir.is_dir(): continue
            name = exp_dir.name
            if exp_name_filter and not any(f in name for f in exp_name_filter): continue
            for seed_dir in sorted(exp_dir.glob('seed_*/lightning_logs/version_*')):
                seed = int(seed_dir.parent.parent.name.split('_')[1])
                metrics_f = seed_dir / 'metrics.csv'
                if not metrics_f.exists(): continue
                try:
                    df = pd.read_csv(metrics_f)
                    df['exp_name'] = name; df['seed'] = seed; df['dataset'] = ds
                    rows.append(df)
                except: pass
    if not rows: print('No metrics.csv found.'); return pd.DataFrame()
    df = pd.concat(rows, ignore_index=True)
    print(f'Learning curves: {len(df)} epoch rows | exps={df.exp_name.nunique()} | seeds={sorted(df.seed.unique())}')
    return df

lc_exp1 = load_learning_curves(EXPERIMENTS_ROOT,
    exp_name_filter=['graph_ensemble_deletion','graph_ensemble_addition','graph_ensemble_reversal'])
lc_exp2 = load_learning_curves(EXPERIMENTS_ROOT,
    exp_name_filter=['gensemble_edge_count_u2c'])

In [ ]:
LOSS_COLS  = ['train_task_loss', 'train_concept_loss', 'train_total_loss']
LOSS_COLOR = {'train_task_loss': '#e74c3c', 'train_concept_loss': '#3498db', 'train_total_loss': '#2c3e50'}

def plot_lc(lc_df, exp_name, title, fname):
    sub = lc_df[lc_df['exp_name'] == exp_name]
    if sub.empty: return
    avail = [c for c in LOSS_COLS if c in sub.columns]
    if not avail: return
    # drop rows where epoch is NaN (val-only rows)
    sub = sub.dropna(subset=['epoch'])

    fig, axes = plt.subplots(1, len(avail), figsize=(5*len(avail), 4), sharey=False)
    if len(avail) == 1: axes = [axes]
    n_seeds = sub['seed'].nunique()
    fig.suptitle(f'{title}\n(mean ± std across {n_seeds} seeds, faint lines = individual seeds)',
                 fontsize=10, fontweight='bold')

    for ax, loss in zip(axes, avail):
        col = LOSS_COLOR[loss]
        # individual seed lines (faint)
        for s, sdf in sub.groupby('seed'):
            ep = sdf.dropna(subset=[loss]).groupby('epoch')[loss].mean()
            ax.plot(ep.index, ep.values, color=col, lw=0.7, alpha=0.3)
        # mean ± std across seeds
        agg = sub.dropna(subset=[loss]).groupby('epoch')[loss].agg(['mean','std']).reset_index()
        ax.plot(agg['epoch'], agg['mean'], color=col, lw=2.5, label='mean')
        ax.fill_between(agg['epoch'], agg['mean']-agg['std'], agg['mean']+agg['std'],
                        alpha=0.2, color=col)
        ax.set_title(loss.replace('train_','').replace('_loss','').replace('_',' '), fontsize=9)
        ax.set_xlabel('Epoch'); ax.set_ylabel('Loss'); ax.tick_params(labelsize=8)

    plt.tight_layout()
    plt.savefig(f'{fname}.png', dpi=150, bbox_inches='tight')
    plt.show()

# ── Exp1: one plot per action/level ──────────────────────────────────────────
if len(lc_exp1) == 0:
    print('No Exp1 learning curves yet.')
else:
    for action in ACTIONS:
        for level in LEVELS:
            exp_name = f'graph_ensemble_{action}_{level}'
            plot_lc(lc_exp1, exp_name,
                    f'Exp1 learning curve — {action}/{level}',
                    f'lc_exp1_{action}_{level}')

In [ ]:
# ── Exp2: all edge counts overlaid on one plot per loss ──────────────────────
if len(lc_exp2) == 0:
    print('No Exp2 learning curves yet.')
else:
    lc_exp2['edge_count'] = lc_exp2['exp_name'].apply(
        lambda x: int(_re.search(r'(\d+)edges', x).group(1)) if _re.search(r'(\d+)edges', x) else None)
    lc_exp2 = lc_exp2.dropna(subset=['edge_count', 'epoch'])
    lc_exp2['edge_count'] = lc_exp2['edge_count'].astype(int)

    avail = [c for c in LOSS_COLS if c in lc_exp2.columns]
    if avail:
        all_counts = sorted(lc_exp2['edge_count'].unique())
        show_counts = [c for c in [12, 15, 17, 19, 22] if c in all_counts] or all_counts
        cmap = plt.cm.RdYlGn

        fig, axes = plt.subplots(1, len(avail), figsize=(5*len(avail), 4.5), sharey=False)
        if len(avail) == 1: axes = [axes]
        fig.suptitle('Exp2: Learning Curves per Edge Count\n'
                     f'(GT=17, each line = mean across seeds)',
                     fontsize=11, fontweight='bold')

        for ax, loss in zip(axes, avail):
            for count in show_counts:
                sub = lc_exp2[lc_exp2['edge_count'] == count].dropna(subset=[loss])
                if sub.empty: continue
                agg = sub.groupby('epoch')[loss].agg(['mean','std']).reset_index()
                norm = all_counts.index(count) / max(len(all_counts)-1, 1)
                col  = cmap(norm)
                lw   = 3 if count == 17 else 1.5
                ls   = '-' if count == 17 else '--'
                ax.plot(agg['epoch'], agg['mean'], color=col, lw=lw, ls=ls,
                        label=f'{count}{"(GT)" if count==17 else ""}')
                ax.fill_between(agg['epoch'], agg['mean']-agg['std'], agg['mean']+agg['std'],
                                alpha=0.08, color=col)
            ax.set_title(loss.replace('train_','').replace('_loss','').replace('_',' '), fontsize=9)
            ax.set_xlabel('Epoch'); ax.set_ylabel('Loss'); ax.tick_params(labelsize=8)
            ax.legend(fontsize=7, ncol=2)

        plt.tight_layout()
        plt.savefig('lc_exp2_edge_count.png', dpi=150, bbox_inches='tight')
        plt.show()

---
# Cross-Experiment Comparison

Direct comparison of all three experiments.


In [ ]:
# Collect one row per experiment type with key metrics
summary_rows = []
if len(exp1_df)>0 and 'test_task_accuracy' in exp1_df.columns:
    for (action,level),grp in exp1_df.groupby(['action','noise_level']):
        summary_rows.append({'experiment':'Noisy DAG','action':action,'level':level,
            'task_acc':grp['test_task_accuracy'].mean(),
            'concept_acc':grp.get('test_concept_accuracy',pd.Series([float('nan')])).mean(),
            'CCI':grp['CCI'].mean() if 'CCI' in grp.columns else float('nan')})
if len(exp3_df)>0 and 'test_task_accuracy' in exp3_df.columns:
    for action,grp in exp3_df.groupby('action'):
        summary_rows.append({'experiment':'Mixed Levels','action':action,'level':'mixed',
            'task_acc':grp['test_task_accuracy'].mean(),
            'concept_acc':grp.get('test_concept_accuracy',pd.Series([float('nan')])).mean(),
            'CCI':grp['CCI'].mean() if 'CCI' in grp.columns else float('nan')})
if len(gt_df)>0:
    summary_rows.append({'experiment':'GT CREAM','action':'—','level':'—',
        'task_acc':gt_df['test_task_accuracy'].mean(),
        'concept_acc':gt_df.get('test_concept_accuracy',pd.Series([float('nan')])).mean(),
        'CCI':gt_df['CCI'].mean() if 'CCI' in gt_df.columns else float('nan')})
if summary_rows:
    print('=== Cross-experiment summary ===')
    display(pd.DataFrame(summary_rows).round(4))
